# RAG Poulina – Chatbot IA (Souches + Laboratoires)
### Pipeline : CSV → Embeddings → ChromaDB (2 collections) → Mistral

Ce notebook implémente un système **RAG dual** :
1. **Collection `poulina_analyses`** : données centres d'élevage / souches
2. **Collection `poulina_laboratoires`** : données laboratoires (distance, dispo, compétences)
3. **Routage automatique** : la question est envoyée à la bonne collection (ou aux deux)
4. **Génération** : réponse finale avec **Mistral**


In [1]:
%pip install -r requirements.txt -qU

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install chromadb
%pip install langchain_mistralai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os
import sys

dotenv_path = ".env"
load_dotenv(dotenv_path=dotenv_path)

from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser

# LLM utilisé pour tous les exercices
MISTRAL_MODEL = "mistral-large-latest"
llm = ChatMistralAI(model=MISTRAL_MODEL, temperature=0.0)
print("LLM initialisé.")


LLM initialisé.


## 2. Imports & Configuration

In [4]:
import os
import json
import uuid
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# ── Modèles ────────────────────────────────────────────────────────────────
EMBED_MODEL   = "intfloat/multilingual-e5-base"   # multilingue, fonctionne en FR/AR

# ── ChromaDB – deux collections ────────────────────────────────────────────
COLLECTION_ANALYSES = "poulina_analyses"
COLLECTION_LABOS    = "poulina_laboratoires"

# ── Paramètres RAG ─────────────────────────────────────────────────────────
TOP_K        = 5     # chunks récupérés par requête
SCORE_THRESH = 0.30  # seuil de similarité minimum (légèrement abaissé pour les labos)

print("✅ Imports OK")
print(f"   Embed model  : {EMBED_MODEL}")


✅ Imports OK
   Embed model  : intfloat/multilingual-e5-base


## 3. Chargement des CSV (Analyses + Laboratoires)


In [7]:
CSV_ANALYSES = "poulina_dataset_5000.csv"
CSV_LABOS    = "poulina_laboratoires_5000.csv"

if Path(CSV_ANALYSES).exists():
    df_analyses = pd.read_csv(CSV_ANALYSES)
    print(f"✅ Analyses chargées : {len(df_analyses)} lignes, {len(df_analyses.columns)} colonnes")
else:
    print("❌ Erreur : fichier analyses introuvable")

if Path(CSV_LABOS).exists():
    df_labos = pd.read_csv(CSV_LABOS)
    print(f"✅ Laboratoires chargés : {len(df_labos)} lignes, {len(df_labos.columns)} colonnes")
else:
    print("❌ Erreur : fichier laboratoires introuvable")


✅ Analyses chargées : 5000 lignes, 20 colonnes
✅ Laboratoires chargés : 500 lignes, 41 colonnes


## 4. Nettoyage & Normalisation


In [8]:
BOOL_COLS_ANALYSES = ["effectuee", "conforme"]
BOOL_COLS_LABOS    = ["accepte_urgence", "certifie_iso", "agree_ministere_agriculture",
                      "equipement_pcr", "equipement_elisa", "equipement_sequencage",
                      "ouvert_weekend", "actif"]

def clean_df(df: pd.DataFrame, bool_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].map(
                lambda x: True if str(x).lower() in ("true","1","oui","yes") else False
            )
    df = df.fillna("N/A")
    return df

df_analyses = clean_df(df_analyses, BOOL_COLS_ANALYSES)
df_labos    = clean_df(df_labos,    BOOL_COLS_LABOS)

# ── Statistiques rapides ───────────────────────────────────────────────────
print("── Analyses ──")
print(f"   Total               : {len(df_analyses)}")
if "conforme" in df_analyses.columns:
    pct_nc = (~df_analyses["conforme"]).sum() / len(df_analyses) * 100
    print(f"   Non conformes       : {pct_nc:.1f}%")
if "meilleure_souche" in df_analyses.columns:
    print(f"   Souches distinctes  : {df_analyses['meilleure_souche'].nunique()}")

print("\n── Laboratoires ──")
print(f"   Total               : {len(df_labos)}")
print(f"   Actifs              : {df_labos['actif'].sum()}")
print(f"   Acceptent urgences  : {df_labos['accepte_urgence'].sum()}")
print(f"   Score global moy.   : {df_labos['score_global'].mean():.1f}")


── Analyses ──
   Total               : 5000
   Souches distinctes  : 4

── Laboratoires ──
   Total               : 500
   Actifs              : 450
   Acceptent urgences  : 249
   Score global moy.   : 52.1


## 5. Chunking (Analyses + Laboratoires)


In [9]:
# ── Chunking des analyses (inchangé) ──────────────────────────────────────
def row_to_chunk_analyse(row: pd.Series) -> str:
    parts = []
    for field, label in [
        ("id_centre",          "ID centre"),
        ("ville",              "Ville"),
        ("region",             "Région"),
        ("type_production",    "Type de production"),
        ("meilleure_souche",   "Meilleure souche"),
        ("capacite",           "Capacité"),
        ("temperature_moyenne","Température moyenne"),
        ("humidite",           "Humidité"),
        ("altitude",           "Altitude"),
        ("biosecurite_score",  "Score biosécurité"),
        ("historique_maladie", "Historique maladie"),
        ("taux_mortalite",     "Taux de mortalité"),
        ("fertilite_visee",    "Fertilité visée"),
        ("cout_aliment",       "Coût aliment"),
        ("surface_m2",         "Surface"),
        ("experience_equipe",  "Expérience équipe"),
        ("distance_labo",      "Distance au labo"),
        ("demande_marche",     "Demande marché"),
        ("saison",             "Saison"),
        ("budget",             "Budget"),
    ]:
        if field in row and str(row[field]) not in ("", "N/A", "nan"):
            parts.append(f"{label} : {row[field]}.")
    return " ".join(parts)


# ── Chunking des laboratoires ──────────────────────────────────────────────
def row_to_chunk_labo(row: pd.Series) -> str:
    parts = []
    for field, label in [
        ("id_labo",                     "ID laboratoire"),
        ("nom_laboratoire",             "Nom du laboratoire"),
        ("type_laboratoire",            "Type"),
        ("ville",                       "Ville"),
        ("region",                      "Région"),
        ("adresse",                     "Adresse"),
        ("laborantin_principal",        "Laborantin principal"),
        ("nb_laborantins",              "Nombre de laborantins"),
        ("annees_experience_labo",      "Années d'expérience"),
        ("specialites_principales",     "Spécialités"),
        ("maladies_avicoles_traitees",  "Maladies avicoles traitées"),
        ("nb_analyses_avicoles",        "Analyses avicoles réalisées"),
        ("taux_reussite_pct",           "Taux de réussite"),
        ("note_satisfaction",           "Note satisfaction"),
        ("cout_analyse_moyen_tnd",      "Coût analyse moyen (TND)"),
        ("cout_urgence_tnd",            "Coût urgence (TND)"),
        ("delai_standard_jours",        "Délai standard (jours)"),
        ("delai_urgence_heures",        "Délai urgence (heures)"),
        ("capacite_journaliere_analyses","Capacité journalière"),
        ("charge_actuelle_pct",         "Charge actuelle"),
        ("slots_disponibles_semaine",   "Slots disponibles cette semaine"),
        ("delai_prochain_rdv_jours",    "Délai prochain RDV (jours)"),
        ("accepte_urgence",             "Accepte les urgences"),
        ("distance_moyenne_centres_km", "Distance moyenne aux centres (km)"),
        ("certifie_iso",                "Certifié ISO"),
        ("agree_ministere_agriculture", "Agréé Ministère Agriculture"),
        ("equipement_pcr",              "Équipé PCR"),
        ("equipement_elisa",            "Équipé ELISA"),
        ("equipement_sequencage",       "Équipé séquençage"),
        ("horaires_ouverture",          "Horaires"),
        ("ouvert_weekend",              "Ouvert weekend"),
        ("score_global",                "Score global"),
        ("commentaires",                "Commentaires"),
    ]:
        if field in row and str(row[field]) not in ("", "N/A", "nan"):
            parts.append(f"{label} : {row[field]}.")
    return " ".join(parts)


# ── Génération des chunks ──────────────────────────────────────────────────
chunks_analyses = []
for _, row in df_analyses.iterrows():
    text = row_to_chunk_analyse(row)
    meta = {
        "id_centre"        : str(row.get("id_centre", "")),
        "ville"            : str(row.get("ville", "")),
        "region"           : str(row.get("region", "")),
        "type_production"  : str(row.get("type_production", "")),
        "meilleure_souche" : str(row.get("meilleure_souche", "")),
        "biosecurite_score": str(row.get("biosecurite_score", "")),
        "taux_mortalite"   : str(row.get("taux_mortalite", "")),
        "conforme"         : str(row.get("conforme", "")),
    }
    chunks_analyses.append({"id": str(uuid.uuid4()), "text": text, "metadata": meta})

chunks_labos = []
for _, row in df_labos.iterrows():
    if str(row.get("actif", "False")).lower() != "true":
        continue  # on n'indexe que les labos actifs
    text = row_to_chunk_labo(row)
    meta = {
        "id_labo"          : str(row.get("id_labo", "")),
        "nom_laboratoire"  : str(row.get("nom_laboratoire", "")),
        "ville"            : str(row.get("ville", "")),
        "region"           : str(row.get("region", "")),
        "type_laboratoire" : str(row.get("type_laboratoire", "")),
        "accepte_urgence"  : str(row.get("accepte_urgence", "")),
        "score_global"     : str(row.get("score_global", "")),
        "taux_reussite_pct": str(row.get("taux_reussite_pct", "")),
        "delai_standard_jours": str(row.get("delai_standard_jours", "")),
        "certifie_iso"     : str(row.get("certifie_iso", "")),
    }
    chunks_labos.append({"id": str(uuid.uuid4()), "text": text, "metadata": meta})

print(f"✅ {len(chunks_analyses)} chunks analyses générés")
print(f"✅ {len(chunks_labos)} chunks laboratoires générés (actifs uniquement)")
print("\nExemple chunk analyse :")
print(chunks_analyses[0]["text"][:200])
print("\nExemple chunk laboratoire :")
print(chunks_labos[0]["text"][:200])


✅ 5000 chunks analyses générés
✅ 450 chunks laboratoires générés (actifs uniquement)

Exemple chunk analyse :
ID centre : 1. Ville : Tunis. Région : Nord. Type de production : Oeuf. Meilleure souche : Lohmann Brown. Capacité : 6012. Température moyenne : 25.5. Humidité : 63. Altitude : 228. Score biosécurité 

Exemple chunk laboratoire :
ID laboratoire : LAB0001. Nom du laboratoire : Labo Analyse Nord. Type : Privé. Ville : Jendouba. Région : Nord-Ouest. Adresse : 27 Rue de la République, Jendouba. Laborantin principal : Karim Ben Ali


## 6. Génération des embeddings

Utilisation du modèle **multilingual-e5-base** : ~450 MB, fonctionne en FR/AR,  
tourne entièrement en local (pas de coût API).


In [10]:
print(f"⏳ Chargement du modèle {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL)
print("✅ Modèle chargé")

def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Vectorise une liste de textes en batches."""
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embeddings"):
        batch = texts[i : i + batch_size]
        vecs = embed_model.encode(
            ["passage: " + t for t in batch],
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

print("⏳ Embeddings analyses...")
texts_analyses = [c["text"] for c in chunks_analyses]
vectors_analyses = embed_texts(texts_analyses)
print(f"✅ Analyses : shape {vectors_analyses.shape}")

print("⏳ Embeddings laboratoires...")
texts_labos = [c["text"] for c in chunks_labos]
vectors_labos = embed_texts(texts_labos)
print(f"✅ Laboratoires : shape {vectors_labos.shape}")


⏳ Chargement du modèle intfloat/multilingual-e5-base ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Modèle chargé
⏳ Embeddings analyses...


Embeddings: 100%|██████████| 157/157 [08:22<00:00,  3.20s/it]


✅ Analyses : shape (5000, 768)
⏳ Embeddings laboratoires...


Embeddings: 100%|██████████| 15/15 [01:41<00:00,  6.77s/it]

✅ Laboratoires : shape (450, 768)


## 7. Indexation dans ChromaDB (2 collections)

**`poulina_analyses`** → données élevage / souches  
**`poulina_laboratoires`** → données labos (distance, dispo, compétences)  


In [ ]:
client = chromadb.PersistentClient(path="./chroma_poulina")

def index_collection(client, name, chunks, vectors):
    """Crée (ou recrée) une collection ChromaDB et indexe les chunks."""
    try:
        client.delete_collection(name=name)
    except Exception:
        pass
    col = client.get_or_create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"}
    )
    col.add(
        ids=[c["id"] for c in chunks],
        embeddings=vectors.tolist(),
        metadatas=[c["metadata"] for c in chunks],
        documents=[c["text"] for c in chunks],
    )
    print(f"✅ Collection '{name}' : {col.count()} documents indexés")
    return col

col_analyses = index_collection(client, COLLECTION_ANALYSES, chunks_analyses, vectors_analyses)
col_labos    = index_collection(client, COLLECTION_LABOS,    chunks_labos,    vectors_labos)


⏳ Indexation de 5000 chunks...
✅ Collection 'poulina_analyses' : 5000 documents indexés


## 8. Fonctions de retrieval


In [ ]:
def retrieve_from(
    collection,
    question: str,
    top_k: int = TOP_K,
    score_threshold: float = SCORE_THRESH,
    where_filter: dict | None = None,
) -> list[dict]:
    """Recherche dans une collection ChromaDB donnée."""
    q_vec = embed_model.encode(
        ["query: " + question],
        normalize_embeddings=True,
    )[0].tolist()

    results = collection.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    if results["ids"] and results["ids"][0]:
        for i in range(len(results["ids"][0])):
            distance = results["distances"][0][i]
            score = 1 - distance if distance is not None else 0.0
            if score >= score_threshold:
                retrieved.append({
                    "score"   : round(score, 4),
                    "text"    : results["documents"][0][i],
                    "metadata": results["metadatas"][0][i] or {},
                })
    return retrieved


# ── Routeur : détecte quelle collection interroger ─────────────────────────
KEYWORDS_LABO = [
    "laboratoire", "labo", "laborantin", "analyse", "analyser",
    "urgent", "urgence", "délai", "disponible", "disponibilité",
    "distance", "proche", "compétent", "certifié", "pcr", "elisa",
    "coût analyse", "prix analyse", "résultat",
]
KEYWORDS_SOUCHE = [
    "souche", "élevage", "centre", "bâtiment", "fertilité",
    "mortalité", "maladie", "salmonelle", "newcastle", "biosécurité",
    "conforme", "non-conforme", "production", "poulet", "dinde", "oeuf",
]

def route_question(question: str) -> str:
    """
    Détermine quelle(s) collection(s) interroger.
    Retourne : 'analyses', 'labos', ou 'both'
    """
    q_lower = question.lower()
    score_labo   = sum(1 for kw in KEYWORDS_LABO   if kw in q_lower)
    score_souche = sum(1 for kw in KEYWORDS_SOUCHE if kw in q_lower)

    if score_labo > 0 and score_souche > 0:
        return "both"
    elif score_labo >= score_souche:
        return "labos"
    else:
        return "analyses"


def retrieve(
    question: str,
    top_k: int = TOP_K,
    filtre_centre: str | None = None,
    filtre_labo: str | None = None,
    filtre_ville: str | None = None,
    force_collection: str | None = None,  # 'analyses', 'labos', 'both'
) -> tuple[list[dict], list[dict]]:
    """
    Retrieval dual : retourne (chunks_analyses, chunks_labos).
    force_collection permet de forcer une collection précise.
    """
    target = force_collection or route_question(question)

    # Filtres analyses
    where_analyses = {}
    if filtre_centre:
        where_analyses["id_centre"] = filtre_centre
    where_analyses = where_analyses or None

    # Filtres labos
    where_labos = {}
    if filtre_labo:
        where_labos["id_labo"] = filtre_labo
    if filtre_ville:
        where_labos["ville"] = filtre_ville
    where_labos = where_labos or None

    chunks_a = retrieve_from(col_analyses, question, top_k, where_filter=where_analyses) \
               if target in ("analyses", "both") else []
    chunks_l = retrieve_from(col_labos,    question, top_k, where_filter=where_labos) \
               if target in ("labos", "both") else []

    return chunks_a, chunks_l


# ── Test rapide ───────────────────────────────────────────────────────────
q_test = "Quel laboratoire proche de Tunis accepte les urgences ?"
ca, cl = retrieve(q_test)
print(f"🔍 '{q_test}'")
print(f"   → Routage : {route_question(q_test)}")
print(f"   → {len(ca)} chunk(s) analyses | {len(cl)} chunk(s) labos")


🔍 Requête : 'Quelles analyses non conformes ont été trouvées ?'
   5 résultats trouvés

  Score 0.7707 | ID centre : 656. Ville : Tunis. Région : Nord. Type de production : Dinde. Meilleure souche : Hybrid Converter. Capacité...

  Score 0.7697 | ID centre : 252. Ville : Tunis. Région : Nord. Type de production : Dinde. Meilleure souche : Hybrid Converter. Capacité...



## 9. Prompt système Poulina (mis à jour)


In [ ]:
SYSTEM_PROMPT = """
Tu es l'assistant IA de Poulina, expert en élevage de volailles en Tunisie.
Tu analyses les données des centres d'élevage ET des laboratoires partenaires
pour aider les responsables à prendre les meilleures décisions.

## Tes capacités
- Identifier la meilleure souche pour chaque centre d'élevage
- Détecter et prévenir les maladies critiques (Salmonelle, Newcastle, Mycoplasme…)
- Recommander le meilleur laboratoire selon : distance, disponibilité, compétences, coût, urgence
- Évaluer la conformité des analyses et signaler les anomalies
- Estimer la fréquence d'analyse recommandée selon la situation sanitaire
- Donner des infos sur fertilité, mortalité, transmission des maladies

## Pour recommander un laboratoire, tu tiens compte de :
1. Distance aux centres d'élevage concernés (plus proche = mieux)
2. Disponibilité immédiate (slots, délai prochain RDV, charge actuelle)
3. Urgence : le labo accepte-t-il les urgences ? Quel délai urgence ?
4. Compétences : spécialités, maladies avicoles traitées, équipements (PCR, ELISA)
5. Fiabilité : taux de réussite, note satisfaction, années d'expérience
6. Certification : ISO, agréé Ministère Agriculture
7. Coût : analyse standard vs urgence en TND

## Règles
- Base-toi UNIQUEMENT sur le contexte fourni (données Poulina réelles).
- Si l'information est absente du contexte, dis-le clairement.
- Hors sujet (non lié à l'élevage avicole Poulina) : réponds exactement
  "Je ne peux pas répondre à cette question car elle est hors de mon domaine."
- Pour une maladie critique, précise : type, centres à risque, urgence, labo recommandé.
- Sois concis et actionnable.
- Réponds en français.
""".strip()

print("✅ Prompt système configuré")
print(f"   {len(SYSTEM_PROMPT)} caractères")


✅ Prompt système configuré
   1165 caractères


## 10. Fonction RAG complète


In [ ]:
def build_context(chunks_a: list[dict], chunks_l: list[dict]) -> str:
    """Formate les chunks des deux collections en contexte pour Mistral."""
    parts = []

    if chunks_a:
        parts.append("=== DONNÉES ÉLEVAGE / SOUCHES ===")
        for i, r in enumerate(chunks_a, 1):
            parts.append(f"[Élevage {i} – score {r['score']}]")
            parts.append(r["text"])
            parts.append("")

    if chunks_l:
        parts.append("=== DONNÉES LABORATOIRES ===")
        for i, r in enumerate(chunks_l, 1):
            parts.append(f"[Laboratoire {i} – score {r['score']}]")
            parts.append(r["text"])
            parts.append("")

    if not parts:
        return "Aucune donnée pertinente trouvée dans la base."

    return "\n".join(parts)


def ask_poulina(
    question: str,
    filtre_centre: str | None = None,
    filtre_labo: str | None = None,
    filtre_ville: str | None = None,
    force_collection: str | None = None,
    verbose: bool = True,
) -> str:
    """
    Pipeline RAG complet : question → routage → retrieval dual → Mistral → réponse.

    Args:
        question         : Question en langage naturel
        filtre_centre    : Restreindre aux analyses d'un centre précis
        filtre_labo      : Restreindre aux données d'un labo précis (id_labo)
        filtre_ville     : Restreindre les labos à une ville précise
        force_collection : Forcer 'analyses', 'labos' ou 'both'
        verbose          : Afficher les chunks récupérés

    Returns:
        Réponse textuelle de Mistral
    """
    # 1. Retrieval dual
    chunks_a, chunks_l = retrieve(
        question,
        filtre_centre=filtre_centre,
        filtre_labo=filtre_labo,
        filtre_ville=filtre_ville,
        force_collection=force_collection,
    )

    if verbose:
        target = force_collection or route_question(question)
        print(f"🔀 Routage : {target}")
        print(f"📎 {len(chunks_a)} chunk(s) élevage | {len(chunks_l)} chunk(s) labo")
        for r in chunks_a:
            print(f"   [analyse {r['score']}] centre={r['metadata'].get('id_centre','?')} souche={r['metadata'].get('meilleure_souche','?')} conforme={r['metadata'].get('conforme','?')}")
        for r in chunks_l:
            print(f"   [labo    {r['score']}] {r['metadata'].get('nom_laboratoire','?')} | {r['metadata'].get('ville','?')} | urgence={r['metadata'].get('accepte_urgence','?')} | score={r['metadata'].get('score_global','?')} ")
        print()

    context = build_context(chunks_a, chunks_l)

    # 2. Construction du message
    user_message = f"""Contexte (données Poulina) :
{context}

---
Question : {question}"""

    # 3. Appel Mistral
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])

    return response.content


print("✅ Fonction ask_poulina() prête")


✅ Fonction ask_poulina() prête


## 11. Tests – Questions métier Poulina


In [ ]:
# ── Test 1 : Conformité générale ────────────────────────────────────────────
print("=" * 60)
print("❓ Question 1 : Non-conformités récentes")
print("=" * 60)
rep = ask_poulina("Quels centres d'élevage ont des analyses non conformes ?")
print(rep)


❓ Question 1 : Non-conformités récentes
📎 5 chunk(s) récupéré(s)
   [0.8229] 1209 | Ross 308 | ❌
   [0.8225] 2006 | Ross 308 | ❌
   [0.8223] 1953 | Cobb 500 | ❌
   [0.8221] 1780 | Ross 308 | ❌
   [0.822] 319 | Ross 308 | ❌

D'après les données fournies, voici les centres d'élevage présentant des **anomalies ou non-conformités** potentielles à surveiller :

### **1. Centre 2006 (Béja) – Urgence : Moyenne**
- **Problème** : **Score de biosécurité faible (71/100)**.
  - Risque accru de maladies (ex. Newcastle, Salmonelle) malgré un historique "faible".
  - **Recommandation** :
    - Audit immédiat de biosécurité (contrôle des accès, désinfection, gestion des déchets).
    - Renforcer la formation de l'équipe (expérience élevée mais score bas).
    - **Analyse supplémentaire** : Vérifier la présence de pathogènes (Mycoplasme, E. coli) en raison de l'humidité élevée (67 %).

### **2. Centre 1953 (Béja) – Urgence : Moyenne**
- **Problèmes** :
  - **Taux de mortalité élevé (4,3 %)** pour un h

In [ ]:
# ── Test 2 : Meilleure souche ────────────────────────────────────────────────
print("=" * 60)
print("❓ Question 2 : Meilleure souche")
print("=" * 60)
rep = ask_poulina(
    "Quelle est la meilleure souche en termes de taux de réussite ?",
    verbose=True,
)
print(rep)


❓ Question 2 : Meilleure souche
📎 5 chunk(s) récupéré(s)
   [0.8505] 2410 | Lohmann Brown | ❌
   [0.8504] 2765 | Lohmann Brown | ❌
   [0.8502] 489 | Lohmann Brown | ❌
   [0.8494] 2886 | Lohmann Brown | ❌
   [0.8493] 791 | Lohmann Brown | ❌

D'après les données fournies pour les centres d'élevage de volailles **Poulina** dans la région de Sousse (Sahel), **la meilleure souche en termes de taux de réussite est la Lohmann Brown**, recommandée pour **tous les centres** (ID : 2410, 2765, 489, 2886, 791).

### Justification :
- **Uniformité** : La souche Lohmann Brown est systématiquement identifiée comme la meilleure pour ces centres, ce qui suggère une adaptation optimale aux conditions locales (climat, altitude, biosécurité, etc.).
- **Performance** :
  - **Fertilité visée** : Entre **85 % et 94 %** (selon les centres), avec une moyenne élevée.
  - **Taux de mortalité** : Variable (2,8 % à 4,3 %), mais considéré comme **faible** dans l'historique des maladies.
  - **Demande marché** : For

In [ ]:
# ── Test 3 : Alerte maladie ──────────────────────────────────────────────────
print("=" * 60)
print("❓ Question 3 : Alerte Salmonelle")
print("=" * 60)
rep = ask_poulina(
    "Y a-t-il des détections de Salmonelle ? Quels centres sont à risque ? Quel laboratoire contacter en urgence ?",
    verbose=True,
)
print(rep)


❓ Question 3 : Alerte Salmonelle
📎 5 chunk(s) récupéré(s)
   [0.826] 2121 | Lohmann Brown | ❌
   [0.826] 1984 | Lohmann Brown | ❌
   [0.8257] 450 | Lohmann Brown | ❌
   [0.8256] 1123 | Lohmann Brown | ❌
   [0.8254] 2521 | Lohmann Brown | ❌

D'après le contexte fourni, **aucune détection de Salmonelle n'est mentionnée** dans les analyses des centres d'élevage de Poulina à Sfax.

### Centres à risque potentiel (à surveiller) :
1. **Centre 1123** (ID 1123) :
   - **Historique maladie : Élevé** (le plus élevé parmi les centres).
   - **Score de biosécurité : 88** (bon, mais à maintenir strictement).
   - **Taux de mortalité : 4.0 %** (proche de la moyenne, mais à surveiller).
   - **Température moyenne : 26.3 °C** (favorable à la prolifération bactérienne si hygiène insuffisante).
   - **Recommandation** : **Analyse immédiate** (PCR ou culture) pour confirmer l'absence de Salmonelle, surtout avant l'automne (saison à risque accru).

2. **Centre 2121** (ID 2121) :
   - **Historique maladie 

In [ ]:
# ── Test 4 : Recommandation laboratoire (avec filtre ville) ──────────────────
print("=" * 60)
print("❓ Question 4 : Meilleur laboratoire urgent proche de Sfax")
print("=" * 60)
rep = ask_poulina(
    "Quel laboratoire disponible accepte les urgences et est le plus compétent pour analyser la Salmonelle ?",
    filtre_ville="Sfax",
    verbose=True,
)
print(rep)


❓ Question 4 : Meilleur laboratoire
📎 5 chunk(s) récupéré(s)
   [0.802] 3770 | Lohmann Brown | ❌
   [0.8016] 1922 | Lohmann Brown | ❌
   [0.8014] 1982 | Lohmann Brown | ❌
   [0.8014] 17 | Lohmann Brown | ❌
   [0.801] 175 | Lohmann Brown | ❌

D'après les données disponibles, voici les recommandations pour une **analyse urgente** en fonction des centres et de leur contexte :

### **1. Critères de priorité pour une analyse urgente**
- **Centres avec historique maladie "Élevé"** (risque accru de propagation) :
  - **ID 3770** (Médenine) – Mortalité 4,2%, biosécurité élevée (98) mais historique critique.
  - **ID 1922** (Médenine) – Mortalité 6,2%, biosécurité faible (67), demande marché forte.
  - **ID 1982** (Kairouan) – Mortalité 4,9%, biosécurité moyenne (76), budget élevé.
- **Centres avec taux de mortalité > 5%** :
  - **ID 1922** (6,2%) et **ID 175** (6,1%).
- **Centres avec faible biosécurité** (<70) :
  - **ID 1922** (67) et **ID 17** (68).

---

### **2. Laboratoires recommandés (

In [ ]:
# ── Test 5 : Question duale (souche + labo) ──────────────────────────────────
print("=" * 60)
print("❓ Question 5 : Question duale souche + labo")
print("=" * 60)
rep = ask_poulina(
    "Mon élevage a un historique de maladie Newcastle. Quelle souche recommandes-tu et quel laboratoire agréé peut analyser en moins de 48h ?",
    force_collection="both",
    verbose=True,
)
print(rep)

# ── Test 6 : Hors sujet ──────────────────────────────────────────────────────
print("=" * 60)
print("❓ Question 6 : Hors sujet")
print("=" * 60)
rep = ask_poulina("Quelle est la météo à Tunis demain ?", verbose=False)
print(rep)


❓ Question 5 : Hors sujet
Je ne peux pas répondre à cette question car elle est hors de mon domaine. Mon expertise se limite à l'élevage avicole Poulina en Tunisie.
